# Zambia v3 — GEE fire exposure (MODIS burned area + FIRMS / VIIRS)

**Purpose:** Pilot **Google Earth Engine** pulls for PI-requested **MODIS burned area** and **FIRMS-class active fire** products, summarized to **admin‑2 (district)** polygons for **Zambia**, on a **monthly** cadence over the **12 months before survey fieldwork**.

**Survey / exposure dates** (same as `zambia_modis_fire_v1.ipynb`): fieldwork **2014-08-04 → 2014-10-05**; exposure window **2013-08-04 → 2014-08-03** (inclusive).

For those months over Zambia, **MCD64A1** and **FIRMS** in Earth Engine return imagery under normal date filters (no empty collections in the pilots below).

**Related:** Hansen forest loss → `zambia_gee_v2.ipynb` + `gee_zambia/hansen_zonal.py`. FIRMS bulk/API → `zambia_modis_fire_v1.ipynb`.

## Catalog — derived fire products in Earth Engine (relevant to 2013–2014 Zambia)

| Earth Engine ID | Product | Cadence | Covers Aug 2013 – Aug 2014? | Notes |
|-----------------|---------|---------|-----------------------------|--------|
| `MODIS/061/MCD64A1` | MODIS **burned area** (v6.1) | Monthly | **Yes** (global since Nov 2000) | `BurnDate` 1–366 = DOY of burn in that month’s grid; 0 = unburned. [Catalog](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD64A1) |
| `FIRMS` | MODIS **active fire** (LANCE, rasterized ~1 km) | Daily | **Yes** (since Nov 2000) | NRT quality; `T21` brightness temp. [Catalog](https://developers.google.com/earth-engine/datasets/catalog/FIRMS) |
| `NASA/VIIRS/002/VNP14A1` | VIIRS **thermal anomalies / fire** (SNPP, L3) | Daily | **Yes** (since Jan 2012) | 1 km; `FireMask` 7/8/9 = fire confidence levels; `MaxFRP`. [Catalog](https://developers.google.com/earth-engine/datasets/catalog/NASA_VIIRS_002_VNP14A1) |
| `NASA/LANCE/SNPP_VIIRS/C2` (and NOAA20) | VIIRS **375 m FIRMS** NRT raster | Daily | **No** for 2013–2014 | EE catalog availability starts **2023**; use for recent tests only, not this VACS window. |

**PI wording:** “MODIS Burned Areas and FIRMS in GEE” → here **MCD64A1** + **`FIRMS`** collection; **VIIRS** for this era is **`NASA/VIIRS/002/VNP14A1`**, not the LANCE 375 m C2 layers.

## §1 — Repo root, `.env`, calendar months

Builds one row per **calendar month** overlapping the exposure window: **`month_start`–`month_end`** (inclusive days) plus **`filter_end_exclusive`** for `ee.Filter.date` (half-open upper bound).

In [ ]:
from __future__ import annotations

import calendar
import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

_repo = next(
    (
        d
        for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (d / "utils" / "repo_paths.py").is_file() and (d / "data" / "raw").is_dir()
    ),
    None,
)
if _repo is None:
    raise RuntimeError("Cannot find repo root (expected utils/repo_paths.py and data/raw/).")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from utils.repo_paths import find_repo_root

ROOT = find_repo_root()
try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass

SURVEY_FIELD_START = date(2014, 8, 4)
EXPOSURE_START = date(2013, 8, 4)
EXPOSURE_END = date(2014, 8, 3)


def month_slices(start: date, end: date) -> list[tuple[date, date]]:
    """Inclusive calendar month slices clipped to [start, end] (both inclusive)."""
    out: list[tuple[date, date]] = []
    y, m = start.year, start.month
    while True:
        ms = date(y, m, 1)
        last = date(y, m, calendar.monthrange(y, m)[1])
        s = max(ms, start)
        e = min(last, end)
        if s <= e:
            out.append((s, date(e.year, e.month, e.day)))
        if (y, m) >= (end.year, end.month):
            break
        if m == 12:
            y += 1
            m = 1
        else:
            m += 1
    return out


def next_day(d: date) -> date:
    from datetime import timedelta

    return d + timedelta(days=1)


months = month_slices(EXPOSURE_START, EXPOSURE_END)
month_rows = []
for s, e in months:
    month_rows.append({"month_start": s, "month_end": e, "filter_end_exclusive": next_day(e)})
months_df = pd.DataFrame(month_rows)
print("ROOT =", ROOT)
print("SURVEY_FIELD_START", SURVEY_FIELD_START, "EXPOSURE", EXPOSURE_START, "..", EXPOSURE_END)
print("n_months", len(months_df))
display(months_df)

## §2 — Earth Engine session

Requires `earthengine authenticate` and a GCP **project id** (`EARTHENGINE_PROJECT` or default in cell).

In [ ]:
import ee

EE_PROJECT = (os.environ.get("EARTHENGINE_PROJECT") or "ipv-exposure-research").strip()
ee.Initialize(project=EE_PROJECT)
print("OK", EE_PROJECT)

## §3 — Zambia admin‑2 (GAUL 2015)

Same boundary source as Hansen zonal (`FAO/GAUL/2015/level2`). Set `USE_SIMPLIFIED = True` if `reduceRegions` times out (coarser geometry).

In [ ]:
USE_SIMPLIFIED = False
asset = "FAO/GAUL_SIMPLIFIED_500m/2015/level2" if USE_SIMPLIFIED else "FAO/GAUL/2015/level2"
regions = ee.FeatureCollection(asset).filter(ee.Filter.eq("ADM0_NAME", "Zambia"))
print("n_features", regions.size().getInfo(), "asset", asset)

## §4 — Prototype: MODIS burned area (MCD64A1) one month → `burn_area_ha` per district

**Definition (v0):** pixels with `BurnDate > 0` in that month’s MCD64 image × `pixelArea()` → sum per polygon → hectares. **Not** yet a PI “normalized burn rate” (e.g. per land area or rain season) — add denominator once agreed.

Edit `DEMO_MONTH_START` / `DEMO_MONTH_END` to probe another month.

In [ ]:
DEMO_MONTH_START = "2013-08-01"
DEMO_MONTH_END_EXCLUSIVE = "2013-09-01"

mcd = (
    ee.ImageCollection("MODIS/061/MCD64A1")
    .filter(ee.Filter.date(DEMO_MONTH_START, DEMO_MONTH_END_EXCLUSIVE))
    .select("BurnDate")
)
burned = mcd.max().gt(0).rename("burn_mask")
area_m2 = burned.multiply(ee.Image.pixelArea()).rename("burn_area_m2")
reduced = area_m2.reduceRegions(
    collection=regions,
    reducer=ee.Reducer.sum(),
    scale=500,
    tileScale=4,
    maxPixelsPerRegion=1e13,
)
info = reduced.getInfo()
rows = []
for f in info.get("features", []):
    p = f.get("properties") or {}
    m2 = p.get("burn_area_m2") or p.get("sum")
    p["burn_area_ha"] = float(m2) / 10000.0 if m2 is not None else None
    rows.append(p)
burn_df = pd.DataFrame(rows).sort_values("burn_area_ha", ascending=False)
print("month", DEMO_MONTH_START[:7], "total burn ha (sum districts)", burn_df["burn_area_ha"].sum())
burn_tbl = burn_df[["ADM1_NAME", "ADM2_NAME", "burn_area_ha"]].sort_values(
    ["ADM1_NAME", "ADM2_NAME"]
).reset_index(drop=True)
print("districts (rows)", len(burn_tbl))
with pd.option_context("display.max_rows", None, "display.width", None, "display.max_columns", None):
    display(burn_tbl)

## §5 — Prototype: FIRMS (MODIS) one month → hot‑pixel **area** proxy per district

**Definition (v0):** daily `FIRMS` `T21` > **325 K**; mosaic **max** over the month; threshold → binary × `pixelArea()` → sum ha. This is a **detection footprint** proxy, not flame area.

In [ ]:
firms = (
    ee.ImageCollection("FIRMS")
    .filter(ee.Filter.date(DEMO_MONTH_START, DEMO_MONTH_END_EXCLUSIVE))
    .select("T21")
)
hot = firms.max().gt(325).rename("hot_mask")
hot_m2 = hot.multiply(ee.Image.pixelArea()).rename("hot_area_m2")
reduced_f = hot_m2.reduceRegions(
    collection=regions,
    reducer=ee.Reducer.sum(),
    scale=1000,
    tileScale=4,
    maxPixelsPerRegion=1e13,
)
info_f = reduced_f.getInfo()
rows_f = []
for f in info_f.get("features", []):
    p = f.get("properties") or {}
    m2 = p.get("hot_area_m2") or p.get("sum")
    p["hot_area_ha"] = float(m2) / 10000.0 if m2 is not None else None
    rows_f.append(p)
firms_df = pd.DataFrame(rows_f).sort_values("hot_area_ha", ascending=False)
print("month", DEMO_MONTH_START[:7], "total hot-area proxy ha", firms_df["hot_area_ha"].sum())
firms_tbl = firms_df[["ADM1_NAME", "ADM2_NAME", "hot_area_ha"]].sort_values(
    ["ADM1_NAME", "ADM2_NAME"]
).reset_index(drop=True)
print("districts (rows)", len(firms_tbl))
with pd.option_context("display.max_rows", None, "display.width", None, "display.max_columns", None):
    display(firms_tbl)

## §6 — Next (not run here)

- Loop **§4–§5** over `months_df` and **export long CSV** under `data/raw/exposure_gee/zambia/`.
- **VIIRS `NASA/VIIRS/002/VNP14A1`:** same pattern (`FireMask` ≥ 7 nominal+ or use `MaxFRP`).
- Define **normalized burn rate** denominator with PI (district land area from GAUL vs static mask).
- **Other VACS countries:** parameterize `ADM0_NAME` + survey dates from `VACS_survey_time.csv` (or PI table).